# NYC TLC Trip Data — Geospatial Analysis

This notebook visualizes NYC taxi trip patterns using H3 hexagonal grids and KeplerGL.

**Prerequisites**: `pip install -r requirements-notebook.txt`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import pyarrow.parquet as pq
from scripts.geo import load_taxi_zones, load_h3_mapping, get_zone_centroids, get_zone_lookup

## 1. Load Trip Data

Load a single month of yellow taxi data and inspect basic stats.

In [ ]:
# Load a month of yellow taxi data — adjust path as needed
table = pq.read_table('../yellow_tripdata_2026-03.parquet')
trips = table.to_pandas()

print(f"Rows: {len(trips):,}")
print(f"Date range: {trips['tpep_pickup_datetime'].min()} to {trips['tpep_pickup_datetime'].max()}")
print(f"\nPULocationID value counts (top 10):")
print(trips['PULocationID'].value_counts().head(10))

## 2. Load Taxi Zones

In [ ]:
import matplotlib.pyplot as plt

zones = load_taxi_zones()
print(f"{len(zones)} taxi zones loaded")

fig, ax = plt.subplots(1, 1, figsize=(12, 10))
zones.plot(column='borough', legend=True, ax=ax, edgecolor='gray', linewidth=0.3)
ax.set_title('NYC Taxi Zones by Borough')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 3. Zone Choropleth — Trip Counts by Pickup Zone

In [ ]:
# Aggregate trip counts by pickup zone
trip_counts = trips['PULocationID'].value_counts().reset_index()
trip_counts.columns = ['LocationID', 'trip_count']

# Merge with zone geometries
zones_with_trips = zones.merge(trip_counts, on='LocationID', how='left')
zones_with_trips['trip_count'] = zones_with_trips['trip_count'].fillna(0)

fig, ax = plt.subplots(1, 1, figsize=(12, 10))
zones_with_trips.plot(
    column='trip_count', legend=True, ax=ax,
    edgecolor='gray', linewidth=0.2,
    cmap='YlOrRd', scheme='quantiles', k=7,
    legend_kwds={'title': 'Trip Count', 'loc': 'lower left'}
)
ax.set_title('Yellow Taxi Pickups by Zone (March 2026)')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4. KeplerGL — Zone Choropleth

In [ ]:
from keplergl import KeplerGl

# Prepare GeoJSON-compatible DataFrame for KeplerGL
kepler_zones = zones_with_trips[['LocationID', 'zone', 'borough', 'trip_count', 'geometry']].copy()

map_zones = KeplerGl(height=600)
map_zones.add_data(data=kepler_zones, name='taxi_zones')
map_zones

## 5. Build / Load H3 Mapping

In [ ]:
h3_df = load_h3_mapping()
print(f"{len(h3_df)} H3 cells across {h3_df['LocationID'].nunique()} zones")
print(f"Resolution: {h3_df['h3_resolution'].iloc[0]}")
print(f"\nCells per zone distribution:")
print(h3_df.groupby('LocationID').size().describe())

## 6. Distribute Trip Counts to H3 Cells

Trips are recorded at zone level. We distribute each zone's trip count
equally across its H3 cells. This is a simplification — real demand
concentrates along streets and near transit — but it produces a smooth
hex heatmap that's more visually informative than irregular zone polygons.

In [ ]:
# Count cells per zone for even distribution
cells_per_zone = h3_df.groupby('LocationID').size().rename('cell_count')

# Merge trip counts with H3 mapping
h3_trips = h3_df.merge(trip_counts, on='LocationID', how='left')
h3_trips['trip_count'] = h3_trips['trip_count'].fillna(0)
h3_trips = h3_trips.merge(cells_per_zone, on='LocationID')

# Distribute zone-level counts uniformly across hex cells
h3_trips['hex_trips'] = h3_trips['trip_count'] / h3_trips['cell_count']

print(f"H3 hex trip data: {len(h3_trips)} rows")
print(f"Total trips distributed: {h3_trips['hex_trips'].sum():,.0f}")
print(h3_trips[['h3_index', 'zone', 'borough', 'hex_trips', 'h3_lat', 'h3_lng']].head(10))

## 7. KeplerGL — H3 Hexbin Heatmap

In [ ]:
hex_data = h3_trips[['h3_index', 'h3_lat', 'h3_lng', 'hex_trips', 'borough', 'zone']].copy()
hex_data = hex_data[hex_data['hex_trips'] > 0]

map_hex = KeplerGl(height=600)
map_hex.add_data(data=hex_data, name='h3_trips')
map_hex

## 8. Origin-Destination Arc Map

Visualize the top routes as arcs between zone centroids.

In [ ]:
# Build OD pairs
od = trips.groupby(['PULocationID', 'DOLocationID']).size().reset_index(name='trip_count')
od = od.sort_values('trip_count', ascending=False).head(50)

# Get centroids
centroids = get_zone_centroids()
zone_lookup = get_zone_lookup()

# Build arc DataFrame
arcs = []
for _, row in od.iterrows():
    pu, do = int(row['PULocationID']), int(row['DOLocationID'])
    if pu in centroids and do in centroids:
        pu_lat, pu_lng = centroids[pu]
        do_lat, do_lng = centroids[do]
        pu_info = zone_lookup.get(pu, {})
        do_info = zone_lookup.get(do, {})
        arcs.append({
            'src_lat': pu_lat, 'src_lng': pu_lng,
            'dst_lat': do_lat, 'dst_lng': do_lng,
            'trip_count': row['trip_count'],
            'route': f"{pu_info.get('zone', pu)} -> {do_info.get('zone', do)}",
        })

arc_df = pd.DataFrame(arcs)
print(f"Top {len(arc_df)} OD pairs")
print(arc_df.head())

map_arcs = KeplerGl(height=600)
map_arcs.add_data(data=arc_df, name='od_arcs')
map_arcs

## 9. Temporal Analysis — Hourly Pickup Demand

In [ ]:
# Compute hourly trip counts by zone
trips['pickup_hour'] = trips['tpep_pickup_datetime'].dt.hour
hourly = trips.groupby(['PULocationID', 'pickup_hour']).size().reset_index(name='trip_count')

# Merge with H3 mapping and distribute
hourly_h3 = h3_df.merge(hourly, on='LocationID', how='inner')
hourly_h3 = hourly_h3.merge(cells_per_zone, on='LocationID')
hourly_h3['hex_trips'] = hourly_h3['trip_count'] / hourly_h3['cell_count']

# Create a synthetic timestamp for KeplerGL time filter
hourly_h3['timestamp'] = pd.to_datetime('2026-03-15') + pd.to_timedelta(hourly_h3['pickup_hour'], unit='h')

time_data = hourly_h3[['h3_index', 'h3_lat', 'h3_lng', 'hex_trips', 'borough', 'timestamp']].copy()
time_data = time_data[time_data['hex_trips'] > 0]

print(f"Temporal hex data: {len(time_data)} rows across 24 hours")

map_time = KeplerGl(height=600)
map_time.add_data(data=time_data, name='hourly_hex')
map_time

## 10. Borough Summary Statistics

In [ ]:
# Join trips with zone info
zone_info = pd.DataFrame.from_dict(get_zone_lookup(), orient='index')
zone_info.index.name = 'LocationID'
zone_info = zone_info.reset_index()

trips_with_borough = trips.merge(zone_info, left_on='PULocationID', right_on='LocationID', how='left')

borough_stats = trips_with_borough.groupby('borough').agg(
    total_trips=('fare_amount', 'count'),
    avg_fare=('fare_amount', 'mean'),
    avg_distance=('trip_distance', 'mean'),
    avg_tip_pct=('tip_amount', lambda x: (x / trips_with_borough.loc[x.index, 'fare_amount'].replace(0, float('nan'))).mean() * 100),
).round(2)

print(borough_stats.sort_values('total_trips', ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
borough_stats.sort_values('total_trips').plot.barh(y='total_trips', ax=axes[0], title='Total Trips', legend=False)
borough_stats.sort_values('avg_fare').plot.barh(y='avg_fare', ax=axes[1], title='Avg Fare ($)', legend=False)
borough_stats.sort_values('avg_distance').plot.barh(y='avg_distance', ax=axes[2], title='Avg Distance (mi)', legend=False)
plt.tight_layout()
plt.show()